In [0]:
# Databricks notebook source
# =============================================================================
# SKILL WORKFORCE READINESS — BRONZE → SILVER + QUARANTINE
# Catalog   : hackathon_ltm
# Bronze    : hackathon_ltm.bronze.sp_skill_readiness                (source)
# Silver    : hackathon_ltm.silver.silver_skill_readiness            (clean)
# Quarantine: hackathon_ltm.quarantine.quarantine_skill_readiness
# =============================================================================

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 0 ▶ IMPORTS & CONFIGURATION
# ─────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, StringType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

CATALOG            = "hackathon_ltm"
BRONZE_SCHEMA      = "bronze"
SILVER_SCHEMA      = "silver"
QUARANTINE_SCHEMA  = "quarantine"

BRONZE_TABLE       = f"{CATALOG}.{BRONZE_SCHEMA}.sp_skill_readiness"
SILVER_TABLE       = f"{CATALOG}.{SILVER_SCHEMA}.silver_skill_readiness"
QUARANTINE_TABLE   = f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_skill_readiness"

print(f"Source      : {BRONZE_TABLE}")
print(f"Silver      : {SILVER_TABLE}")
print(f"Quarantine  : {QUARANTINE_TABLE}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 1 ▶ READ FROM BRONZE DELTA TABLE
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 1: Reading Bronze Delta Table")
print("="*65)

df_raw = spark.table(BRONZE_TABLE)
total_raw = df_raw.count()

print(f"  Total rows in bronze table : {total_raw}")
df_raw.printSchema()
df_raw.show(5, truncate=False)

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 2 ▶ SCHEMA GUARD — RETAIN ONLY EXPECTED COLUMNS
#
#  Defensive step: ensures any extra columns produced during CSV
#  ingestion (e.g., trailing commas) are silently dropped.
#  Only the 6 canonical columns are carried forward.
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 2: Schema Guard — Retaining Expected Columns Only")
print("="*65)

EXPECTED_COLS = [
    "skill_id", "employee_id",
    "primary_skill", "secondary_skill",
    "skills_declared", "skills_verified"
]

df = df_raw.select(*[c for c in EXPECTED_COLS if c in df_raw.columns])
print(f"  Columns retained : {df.columns}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 3 ▶ BASIC HYGIENE — TRIM WHITESPACE & NULL-COERCE BLANKS
#
#  CSV ingestion often leaves leading/trailing spaces and encodes
#  missing values as empty strings ("").
#  This step:
#    • Trims all string columns
#    • Converts empty strings to NULL (proper SQL NULL semantics)
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 3: Trim Whitespace & Coerce Empty Strings to NULL")
print("="*65)

for col in df.columns:
    df = df.withColumn(col,
        F.when(F.trim(F.col(col)) == "", None)
         .otherwise(F.trim(F.col(col)))
    )

print("  ✅ All columns trimmed; empty strings → NULL")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 4 ▶ NORMALISE skills_declared & skills_verified
#
#  ROOT CAUSE: Both boolean-like columns contain mixed-case
#  variants of the same value:
#    "Yes", "yes", "Y", "YES"  →  "Yes"
#    "No", "no", "N"           →  "No"
#    NULL                      →  NULL  (preserved)
#
#  A single canonical form is required for KPI aggregation
#  (e.g., % declared, % verified) and for boolean derivations.
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 4: Normalise skills_declared & skills_verified to Yes/No")
print("="*65)

def normalise_yn(col_name: str):
    """Map all Yes/Y/YES variants → 'Yes', No/N → 'No', else NULL."""
    return (
        F.when(F.upper(F.col(col_name)).isin("YES", "Y"), "Yes")
         .when(F.upper(F.col(col_name)).isin("NO",  "N"), "No")
         .otherwise(None)  # unexpected value → NULL → quarantine via RULE 5
    )

df = (df
      .withColumn("skills_declared", normalise_yn("skills_declared"))
      .withColumn("skills_verified", normalise_yn("skills_verified"))
)

print("  skills_declared distribution after normalisation:")
df.groupBy("skills_declared").count().show()

print("  skills_verified distribution after normalisation:")
df.groupBy("skills_verified").count().show()

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 5 ▶ STANDARDISE primary_skill (Title-Case & Typo Fix)
#
#  Two types of issues found in primary_skill:
#  A) Case inconsistency : "pyspark" vs "PySpark"
#  B) Value typo         : "powerbi" vs "Power BI"
#
#  Approach:
#    1. Apply a curated correction map for known typos/aliases
#       (case-insensitive lookup).
#    2. Any value NOT in the correction map keeps its original
#       value after Title-Casing as a safe fallback.
#
#  Why not just Title-Case everything?
#    → "PySpark", "HTML/CSS", "ETL", "AWS", "SAP" are proper
#      abbreviations that Title-Case would mangle.
#    → Explicit map is safer for skill taxonomies.
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 5: Standardise primary_skill (Typo Correction)")
print("="*65)

# Correction map: lower(raw_value) → canonical_value
PRIMARY_SKILL_CORRECTIONS = {
    "pyspark"  : "PySpark",
    "powerbi"  : "Power BI",
    # extend this map as new raw values are discovered
}

ps_corrected = F.col("primary_skill")  # default: keep original
for raw_val, canonical_val in PRIMARY_SKILL_CORRECTIONS.items():
    ps_corrected = F.when(
        F.lower(F.col("primary_skill")) == raw_val, canonical_val
    ).otherwise(ps_corrected)

df = df.withColumn("primary_skill", ps_corrected)

print(f"  Correction map applied: {PRIMARY_SKILL_CORRECTIONS}")
print("  primary_skill distinct values after fix:")
df.select("primary_skill").distinct().orderBy("primary_skill").show(40, truncate=False)

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 6 ▶ STANDARDISE secondary_skill (Typo Fix)
#
#  Issue found: "data bricks" (with space) instead of "Databricks"
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 6: Standardise secondary_skill (Typo Correction)")
print("="*65)

SECONDARY_SKILL_CORRECTIONS = {
    "data bricks" : "Databricks",
    "powerbi"     : "Power BI",
    # extend as needed
}

ss_corrected = F.col("secondary_skill")
for raw_val, canonical_val in SECONDARY_SKILL_CORRECTIONS.items():
    ss_corrected = F.when(
        F.lower(F.col("secondary_skill")) == raw_val, canonical_val
    ).otherwise(ss_corrected)

df = df.withColumn("secondary_skill", ss_corrected)
print(f"  Correction map applied: {SECONDARY_SKILL_CORRECTIONS}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 7 ▶ QUARANTINE FLAG LOGIC
#
#  Each row is tagged with a quarantine_reason.
#  A row is sent to QUARANTINE if ANY rule fires.
#  Clean rows (no rule fired) proceed to SILVER.
#
#  RULE 1 — NULL primary key
#            skill_id IS NULL
#
#  RULE 2 — NULL business key
#            employee_id IS NULL
#
#  RULE 3 — NULL core dimension fields
#            primary_skill IS NULL OR secondary_skill IS NULL
#
#  RULE 4 — Invalid ID format
#            skill_id  NOT matching ^SK\d{3,}$
#            employee_id NOT matching ^E\d{5}$
#
#  RULE 5 — Invalid / unmappable skills_declared value
#            After normalisation, if skills_declared is still NULL
#            it means the source value was not Yes/No/Y/N
#
#  RULE 6 — skills_verified NULL when skills_declared = 'Yes'
#            Declared but never verified = incomplete record;
#            cannot compute verified rate KPI accurately.
#            (declared=No with null verified is acceptable —
#             no obligation to verify undeclared skills)
#
#  RULE 7 — Logical contradiction: verified='Yes' but declared='No'
#            Impossible state: cannot verify what was not declared.
#
#  RULE 8 — primary_skill == secondary_skill (exact match after
#            normalisation; indicates data entry error)
#
#  RULE 9 — Duplicate skill_id (PK violation)
#            Keep first occurrence; quarantine all subsequent.
#
#  RULE 10 — Exact duplicate employee + primary_skill + secondary_skill
#            (fully redundant record — same employee, same skill combo)
#            Keep first occurrence; quarantine all subsequent.
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 7: Applying Quarantine Flag Rules")
print("="*65)

# ── RULE 1: NULL primary key
r1 = F.when(F.col("skill_id").isNull(),
            "RULE1_NULL_SKILL_ID")

# ── RULE 2: NULL business key
r2 = F.when(F.col("employee_id").isNull(),
            "RULE2_NULL_EMPLOYEE_ID")

# ── RULE 3: NULL skill dimension fields
r3 = F.when(
    F.col("primary_skill").isNull() | F.col("secondary_skill").isNull(),
    "RULE3_NULL_PRIMARY_OR_SECONDARY_SKILL"
)

# ── RULE 4: Invalid ID format
r4 = F.when(
    (~F.col("skill_id").rlike(r"^SK\d{3,}$")) |
    (~F.col("employee_id").rlike(r"^E\d{5}$")),
    "RULE4_INVALID_ID_FORMAT"
)

# ── RULE 5: skills_declared is NULL after normalisation
#    (original value was not a recognised Yes/No/Y/N variant)
r5 = F.when(
    F.col("skills_declared").isNull(),
    "RULE5_INVALID_SKILLS_DECLARED_VALUE"
)

# ── RULE 6: skills_declared='Yes' but skills_verified is NULL
r6 = F.when(
    (F.col("skills_declared") == "Yes") & F.col("skills_verified").isNull(),
    "RULE6_DECLARED_YES_BUT_VERIFIED_NULL"
)

# ── RULE 7: Logical contradiction — verified='Yes' but declared='No'
r7 = F.when(
    (F.col("skills_verified") == "Yes") & (F.col("skills_declared") == "No"),
    "RULE7_VERIFIED_YES_BUT_DECLARED_NO"
)

# ── RULE 8: primary_skill == secondary_skill
r8 = F.when(
    F.lower(F.trim(F.col("primary_skill"))) ==
    F.lower(F.trim(F.col("secondary_skill"))),
    "RULE8_PRIMARY_EQUALS_SECONDARY_SKILL"
)

# ── RULE 9: Duplicate skill_id (PK) — keep first by skill_id sort
w_sk = Window.partitionBy("skill_id").orderBy("skill_id")
df = df.withColumn("_rn_skillid", F.row_number().over(w_sk))
r9 = F.when(F.col("_rn_skillid") > 1,
            "RULE9_DUPLICATE_SKILL_ID")

# ── RULE 10: Duplicate employee + primary_skill + secondary_skill
#   (exact skill-combo duplicate for same employee)
w_emp = Window.partitionBy(
    "employee_id",
    F.lower(F.trim(F.col("primary_skill"))),
    F.lower(F.trim(F.col("secondary_skill")))
).orderBy("skill_id")

# NOTE: partitionBy with expressions needs a derived column approach
df = (df
      .withColumn("_ps_lower", F.lower(F.trim(F.col("primary_skill"))))
      .withColumn("_ss_lower", F.lower(F.trim(F.col("secondary_skill"))))
)
w_emp2 = Window.partitionBy("employee_id", "_ps_lower", "_ss_lower").orderBy("skill_id")
df = df.withColumn("_rn_empskill", F.row_number().over(w_emp2))
r10 = F.when(F.col("_rn_empskill") > 1,
             "RULE10_DUPLICATE_EMPLOYEE_SKILL_COMBO")

# ── Combine all rules — first match wins (priority order)
df = df.withColumn("quarantine_reason",
    F.coalesce(r1, r2, r3, r4, r5, r6, r7, r8, r9, r10)
)

# Print quarantine breakdown
print("  Quarantine rule breakdown:")
(df.filter(F.col("quarantine_reason").isNotNull())
   .groupBy("quarantine_reason").count()
   .orderBy("count", ascending=False)
   .show(truncate=False))

total_clean_count      = df.filter(F.col("quarantine_reason").isNull()).count()
total_quarantine_count = df.filter(F.col("quarantine_reason").isNotNull()).count()
print(f"  ✅ Clean rows      : {total_clean_count}")
print(f"  🚨 Quarantine rows : {total_quarantine_count}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 8 ▶ SPLIT INTO CLEAN + QUARANTINE DATAFRAMES
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 8: Splitting into Clean and Quarantine DataFrames")
print("="*65)

INTERNAL_COLS = ["_rn_skillid", "_rn_empskill", "_ps_lower", "_ss_lower"]

df_quarantine = (df
    .filter(F.col("quarantine_reason").isNotNull())
    .drop(*INTERNAL_COLS)
    .withColumn("quarantine_timestamp", F.current_timestamp())
    .withColumn("source_table",         F.lit(BRONZE_TABLE))
)

df_clean = (df
    .filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason", *INTERNAL_COLS)
)

print(f"  Clean DataFrame      rows : {df_clean.count()}")
print(f"  Quarantine DataFrame rows : {df_quarantine.count()}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 9 ▶ SILVER TRANSFORMATIONS & DERIVED COLUMNS
#
#  9a. skill_readiness_status  (core KPI label)
#       • "Fully Ready"        — declared=Yes AND verified=Yes
#       • "Declared Not Verified" — declared=Yes, verified=No
#       • "Not Declared"       — declared=No, verified=No/NULL
#       • "Unknown"            — fallback
#
#  9b. is_verified (boolean)
#       — TRUE if skills_verified = 'Yes'
#
#  9c. is_declared (boolean)
#       — TRUE if skills_declared = 'Yes'
#
#  9d. skill_category
#       Maps each primary_skill to a business domain bucket
#       for KPI roll-ups:
#         • "Data & Analytics"     : PySpark, Databricks, SQL, ETL,
#                                    Delta Lake, Data Engineering,
#                                    Data Science, NLP, Deep Learning,
#                                    Machine Learning
#         • "Cloud & DevOps"       : Azure, AWS, Cloud Architecture,
#                                    Cloud Native, Terraform, DevOps,
#                                    CI/CD
#         • "Visualization & BI"   : Power BI, Tableau, Power Apps
#         • "Programming"          : Python, Java, JavaScript, React,
#                                    HTML/CSS, Full Stack
#         • "Enterprise Apps"      : SAP, Salesforce, ERP
#         • "Domain / Industry"    : Healthcare Analytics,
#                                    Retail Insights, Supply Chain
#         • "Other"                : fallback
#
#  9e. verification_gap
#       — "Yes" if declared=Yes but verified=No  (skill gap risk)
#       — "No"  otherwise
#
#  9f. skill_combo
#       Concatenated string: primary_skill + " | " + secondary_skill
#       Useful for deduplication checks and combo-level KPIs.
#
#  9g. Metadata columns
#       ingestion_timestamp, source_table, processing_date
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 9: Silver Transformations & Derived Columns")
print("="*65)

# 9a — skill_readiness_status
df_silver = df_clean.withColumn(
    "skill_readiness_status",
    F.when(
        (F.col("skills_declared") == "Yes") & (F.col("skills_verified") == "Yes"),
        "Fully Ready"
    ).when(
        (F.col("skills_declared") == "Yes") & (F.col("skills_verified") == "No"),
        "Declared Not Verified"
    ).when(
        F.col("skills_declared") == "No",
        "Not Declared"
    ).otherwise("Unknown")
)
print("  ✅ 9a: skill_readiness_status derived")

# 9b — is_verified
df_silver = df_silver.withColumn(
    "is_verified",
    F.when(F.col("skills_verified") == "Yes", True)
     .when(F.col("skills_verified") == "No",  False)
     .otherwise(None)
)
print("  ✅ 9b: is_verified derived")

# 9c — is_declared
df_silver = df_silver.withColumn(
    "is_declared",
    F.when(F.col("skills_declared") == "Yes", True)
     .when(F.col("skills_declared") == "No",  False)
     .otherwise(None)
)
print("  ✅ 9c: is_declared derived")

# 9d — skill_category
DATA_ANALYTICS_SKILLS   = ["PySpark","Databricks","SQL","ETL","Delta Lake",
                            "Data Engineering","Data Science","NLP",
                            "Deep Learning","Machine Learning"]
CLOUD_DEVOPS_SKILLS     = ["Azure","AWS","Cloud Architecture","Cloud Native",
                            "Terraform","DevOps","CI/CD"]
VIZ_BI_SKILLS           = ["Power BI","Tableau","Power Apps"]
PROGRAMMING_SKILLS      = ["Python","Java","JavaScript","React",
                            "HTML/CSS","Full Stack"]
ENTERPRISE_APP_SKILLS   = ["SAP","Salesforce","ERP"]
DOMAIN_SKILLS           = ["Healthcare Analytics","Retail Insights","Supply Chain"]

df_silver = df_silver.withColumn(
    "skill_category",
    F.when(F.col("primary_skill").isin(DATA_ANALYTICS_SKILLS),  "Data & Analytics")
     .when(F.col("primary_skill").isin(CLOUD_DEVOPS_SKILLS),    "Cloud & DevOps")
     .when(F.col("primary_skill").isin(VIZ_BI_SKILLS),          "Visualization & BI")
     .when(F.col("primary_skill").isin(PROGRAMMING_SKILLS),     "Programming")
     .when(F.col("primary_skill").isin(ENTERPRISE_APP_SKILLS),  "Enterprise Apps")
     .when(F.col("primary_skill").isin(DOMAIN_SKILLS),          "Domain / Industry")
     .otherwise("Other")
)
print("  ✅ 9d: skill_category derived")

# 9e — verification_gap flag
df_silver = df_silver.withColumn(
    "verification_gap",
    F.when(
        (F.col("skills_declared") == "Yes") & (F.col("skills_verified") == "No"),
        "Yes"
    ).otherwise("No")
)
print("  ✅ 9e: verification_gap derived")

# 9f — skill_combo (primary | secondary)
df_silver = df_silver.withColumn(
    "skill_combo",
    F.concat_ws(" | ", F.col("primary_skill"), F.col("secondary_skill"))
)
print("  ✅ 9f: skill_combo derived")

# 9g — metadata
df_silver = (df_silver
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_table",        F.lit(BRONZE_TABLE))
    .withColumn("processing_date",     F.current_date())
)
print("  ✅ 9g: Metadata columns added")

print("\n  Final Silver Schema:")
df_silver.printSchema()

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 10 ▶ FINAL COLUMN ORDERING
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 10: Final Column Ordering")
print("="*65)

SILVER_COLUMNS = [
    # ── Keys
    "skill_id", "employee_id",
    # ── Raw skill fields
    "primary_skill", "secondary_skill",
    # ── Raw boolean fields (normalised)
    "skills_declared", "skills_verified",
    # ── Derived status & flags
    "skill_readiness_status", "is_declared", "is_verified",
    "verification_gap",
    # ── Dimension shortcuts
    "skill_category", "skill_combo",
    # ── Metadata
    "ingestion_timestamp", "source_table", "processing_date"
]

df_silver = df_silver.select(*SILVER_COLUMNS)
print(f"  Silver columns ({len(SILVER_COLUMNS)}): {SILVER_COLUMNS}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 11 ▶ DATA QUALITY SUMMARY (pre-write)
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 11: Pre-Write Data Quality Summary")
print("="*65)

print("\n  [ Silver ] skill_readiness_status distribution:")
df_silver.groupBy("skill_readiness_status").count().orderBy("count", ascending=False).show()

print("\n  [ Silver ] skill_category distribution:")
df_silver.groupBy("skill_category").count().orderBy("count", ascending=False).show()

print("\n  [ Silver ] verification_gap distribution:")
df_silver.groupBy("verification_gap").count().show()

print("\n  [ Silver ] top 10 primary_skill:")
df_silver.groupBy("primary_skill").count().orderBy("count", ascending=False).show(10)

print("\n  [ Silver ] Sample rows:")
df_silver.show(5, truncate=False)

print("\n  [ Quarantine ] quarantine_reason distribution:")
df_quarantine.groupBy("quarantine_reason").count().orderBy("count", ascending=False).show(truncate=False)

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 12 ▶ CREATE TARGET SCHEMAS & WRITE SILVER TABLE
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 12: Creating Schemas & Writing Silver Table")
print("="*65)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{QUARANTINE_SCHEMA}")
print("  ✅ Schemas verified / created")

if spark.catalog.tableExists(SILVER_TABLE):
    print(f"  Silver table exists — performing MERGE upsert on skill_id")
    silver_delta = DeltaTable.forName(spark, SILVER_TABLE)
    silver_delta.alias("target").merge(
        df_silver.alias("source"),
        "target.skill_id = source.skill_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print(f"  ✅ MERGE complete into {SILVER_TABLE}")
else:
    print(f"  Silver table does not exist — creating via WRITE")
    (df_silver.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"  ✅ Silver table created : {SILVER_TABLE}")

spark.sql(f"OPTIMIZE {SILVER_TABLE} ZORDER BY (employee_id, primary_skill)")
print(f"  ✅ OPTIMIZE + ZORDER applied on {SILVER_TABLE}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 13 ▶ WRITE QUARANTINE TABLE
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 13: Writing Quarantine Table")
print("="*65)

(df_quarantine.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)
print(f"  ✅ Quarantine table written (APPEND): {QUARANTINE_TABLE}")

# COMMAND ----------
# ─────────────────────────────────────────────────────────────────
# STEP 14 ▶ FINAL RECONCILIATION COUNTS
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("STEP 14: Final Reconciliation")
print("="*65)

silver_count      = spark.table(SILVER_TABLE).count()
quarantine_count  = spark.table(QUARANTINE_TABLE).count()

print(f"  Bronze (source)          : {total_raw:>6} rows")
print(f"  Silver (clean)           : {silver_count:>6} rows")
print(f"  Quarantine               : {quarantine_count:>6} rows")
print(f"  Silver + Quarantine      : {silver_count + quarantine_count:>6} rows  "
      f"({'✅ BALANCED' if silver_count + quarantine_count == total_raw else '❌ MISMATCH — investigate'})")

print("\n" + "="*65)
print("✅  PIPELINE COMPLETE")
print("="*65)

# COMMAND ----------
# =============================================================================
# KPI VALIDATION QUERIES  (run immediately after pipeline to verify silver)
# =============================================================================

# ── 1. Overall Workforce Readiness Rate
spark.sql(f"""
    SELECT
        COUNT(*)                                                        AS total_employees,
        SUM(CAST(is_declared AS INT))                                   AS total_declared,
        SUM(CAST(is_verified AS INT))                                   AS total_verified,
        ROUND(SUM(CAST(is_declared AS INT)) * 100.0 / COUNT(*), 2)     AS declaration_rate_pct,
        ROUND(SUM(CAST(is_verified AS INT)) * 100.0 / COUNT(*), 2)     AS verification_rate_pct
    FROM {SILVER_TABLE}
""").show()

# ── 2. Readiness Status by Skill Category
spark.sql(f"""
    SELECT skill_category,
           skill_readiness_status,
           COUNT(*) AS employee_count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY skill_category), 2) AS pct_within_category
    FROM {SILVER_TABLE}
    GROUP BY skill_category, skill_readiness_status
    ORDER BY skill_category, employee_count DESC
""").show(40, truncate=False)

# ── 3. Verification Gap by Primary Skill (top 15 at-risk skills)
spark.sql(f"""
    SELECT primary_skill,
           skill_category,
           COUNT(*)                                                        AS total,
           SUM(CASE WHEN verification_gap = 'Yes' THEN 1 ELSE 0 END)      AS gap_count,
           ROUND(SUM(CASE WHEN verification_gap = 'Yes' THEN 1 ELSE 0 END)
                 * 100.0 / COUNT(*), 2)                                    AS gap_rate_pct
    FROM {SILVER_TABLE}
    GROUP BY primary_skill, skill_category
    HAVING gap_count > 0
    ORDER BY gap_rate_pct DESC
    LIMIT 15
""").show(truncate=False)

# ── 4. Employees with Multiple Skill Records (for dim_employee_skill)
spark.sql(f"""
    SELECT employee_id, COUNT(*) AS skill_record_count
    FROM {SILVER_TABLE}
    GROUP BY employee_id
    HAVING skill_record_count > 1
    ORDER BY skill_record_count DESC
""").show(10)

# ── 5. Top Skill Combos across workforce
spark.sql(f"""
    SELECT skill_combo, COUNT(*) AS frequency
    FROM {SILVER_TABLE}
    WHERE skill_readiness_status = 'Fully Ready'
    GROUP BY skill_combo
    ORDER BY frequency DESC
    LIMIT 10
""").show(truncate=False)

Source      : hackathon_ltm.bronze.sp_skill_readiness
Silver      : hackathon_ltm.silver.silver_skill_readiness
Quarantine  : hackathon_ltm.quarantine.quarantine_skill_readiness

STEP 1: Reading Bronze Delta Table
  Total rows in bronze table : 500
root
 |-- skill_id: string (nullable = true)
 |-- employee_id: string (nullable = true)
 |-- primary_skill: string (nullable = true)
 |-- secondary_skill: string (nullable = true)
 |-- skills_declared: string (nullable = true)
 |-- skills_verified: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _source_file: string (nullable = true)

+--------+-----------+----------------+---------------+---------------+---------------+--------------------------+--------------+----------------------------------------------------------------------------+
|skill_id|employee_id|primary_skill   |secondary_skill|skills_declared|skills_verified|_ingestion_timestamp      |_source_sy